# Ranker Demo: BM25 + Semantic + Priors

Run the cells in order to test the `ranker` module using sample documents.

This notebook demonstrates:
- Lexical ranking with BM25
- Optional semantic influence via cosine similarity
- Popularity and recency priors
- k edge cases and simple metrics (NDCG@K, MRR)


In [6]:
import os, sys, platform
sys.path.append(os.path.abspath('../src'))
from ranker import rank, ndcg_at_k, mrr
from data.sample_data import get_sample_documents
print(f'Python: {platform.python_version()}')


Python: 3.12.8


In [13]:
docs = get_sample_documents()
len(docs), docs


(5,
 [{'id': 'doc_001',
   'text': 'Blue resume template modern professional',
   'tokens': ['blue', 'resume', 'template', 'modern', 'professional'],
   'clicks': 120,
   'age_days': 10},
  {'id': 'doc_002',
   'text': 'Wedding invitation floral theme',
   'tokens': ['wedding', 'invitation', 'floral', 'theme'],
   'clicks': 80,
   'age_days': 5},
  {'id': 'doc_003',
   'text': 'Business card minimalist design',
   'tokens': ['business', 'card', 'minimalist', 'design'],
   'clicks': 50,
   'age_days': 20},
  {'id': 'doc_004',
   'text': 'Resume template clean layout',
   'tokens': ['resume', 'template', 'clean', 'layout'],
   'clicks': 200,
   'age_days': 60},
  {'id': 'doc_005',
   'text': 'Birthday invitation fun colorful',
   'tokens': ['birthday', 'invitation', 'fun', 'colorful'],
   'clicks': 30,
   'age_days': 2}])

### Lexical ranking (BM25)


In [8]:
results_lex = rank('resume template', docs, k=5)
results_lex


[('doc_004', 1.5049076706454487),
 ('doc_001', 1.3233429194476165),
 ('doc_002', -0.7175853414141935),
 ('doc_003', -1.03998652019166),
 ('doc_005', -1.070678728487212)]

### Priors-only behavior (empty query)


In [9]:
results_priors = rank('', docs, k=3)
results_priors


[('doc_001', 0.2070480494988304),
 ('doc_004', 0.1771933177518169),
 ('doc_002', 0.09708439953327919)]

### Semantic influence (when embeddings are provided)


In [10]:
docs_sem = [
    {'id': 'a', 'tokens': ['irrelevant'], 'emb': [1.0, 0.0]},
    {'id': 'b', 'tokens': ['irrelevant'], 'emb': [0.0, 1.0]},
]
results_sem = rank('', docs_sem, k=2, weights=(0.0, 1.0, 0.0), query_embedding=[1.0, 0.0])
results_sem


[('a', 1.0), ('b', -1.0)]

### k edge cases


In [11]:
rank('resume', docs, k=0), len(rank('resume', docs, k=999))


([], 5)

### Simple metrics (NDCG@K, MRR)


In [12]:
gains = [3.0, 2.0, 1.0]
ndcg_at_k(gains, 3), mrr([0, 0, 1, 0])


(1.0, 0.3333333333333333)

In [1]:
import sys
import os
import time
from typing import List
os.environ.update({
        "OMP_NUM_THREADS": "1",
        "MKL_NUM_THREADS": "1", 
        "NUMEXPR_NUM_THREADS": "1",
        "OPENBLAS_NUM_THREADS": "1",
        "CUDA_VISIBLE_DEVICES": "",
        "TOKENIZERS_PARALLELISM": "false",
        "TRANSFORMERS_OFFLINE": "1",
        "HF_HUB_OFFLINE": "1"
    })

In [4]:
import torch

import transformers
import sentence_transformers
import clip
print('✅ All deep learning packages imported successfully!')

✅ All deep learning packages imported successfully!


In [1]:
import torch